# DiscoCH: near real time vs. retrospective
Validates and compares the two runs of a season, side by side:

- **NRT** (`DiscoCH_NRT.ipynb`): each date interpolated from the observations up to 5 days after it, stepwise min/max of the NRT series.
- **Retro** (`DiscoCH_Retro.ipynb`): each date interpolated from all observations, stepwise min/max of the retrospective series.
- optionally **Retro (NRT min/max)** (`DiscoCH_Retro_NRTminmax.ipynb`): all-data interpolation with the NRT min/max — set `include_nrtminmax = True`.

Contents: the point validation and opportunistic-mapping recall tables, the NRT / Retro / 2018 time series and their difference, the per-region comparison, and a single-pixel interpolation example. Run the pipelines first. Outputs: `output/DiscoCH/<year>/nrt_vs_retro/`.

In [ ]:
# User defined parameters
# Paths are machine-specific and come from config.yaml in the project root
# (copy config.example.yaml to config.yaml and set the paths for your machine).
import calendar
from datetime import date
from src.disco_ch_operational.config import load_config

cfg = load_config()

year = 2026                    # season of both runs (the validation data below are from 2026)
threshold = 4000               # stored value (0-10000): 4000 = 40 % probability
include_nrtminmax = False      # also show the Retro run with the NRT min/max (DiscoCH_Retro_NRTminmax)

# Earlier season to compare with (Bloom et al. 2026 mosaics)
compare_year = 2018
mosaics_compare = cfg.require("mosaics_2018")
compare_pattern = "Proba_mosaic_*.tif"

In [ ]:
# Libraries
import os
import pandas as pd
import matplotlib.pyplot as plt

from src.disco_ch_operational.config import run_dirs
from src.disco_ch_operational.QAQC import point_validation_table, recall_table
from src.disco_ch_operational.compare import plot_year_comparison, rc_params, BIOGEO_NAMES_EN, BIOGEO_COLORS
from src.disco_ch_operational.visualize_interpolation import plot_pixel

# Standard light theme for plots, white figures
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

districts = cfg.require("districts")            # political districts (cantons: field ktnr)
biogeo_regions = cfg.require("biogeo_regions")  # biogeographic regions (field DERegionNa)

# Folders of the runs and of this comparison
nrt_dirs = run_dirs(cfg["output_dir"], year, "nrt")
retro_dirs = run_dirs(cfg["output_dir"], year, "retro")
nrtmm_dirs = run_dirs(cfg["output_dir"], year, "retro_nrtminmax")
out = run_dirs(cfg["output_dir"], year, "nrt_vs_retro")

# The maps compared, and their names in tables and plots
maps = {"NRT": nrt_dirs["masked"], "Retro": retro_dirs["masked"]}
if include_nrtminmax:
    maps["Retro (NRT min/max)"] = nrtmm_dirs["masked"]
for name, path in maps.items():
    print(f"{name:20s} {path} ({'found' if os.path.isdir(path) else 'MISSING: run its notebook first'})")

## Validation
### Point validation
The labelled stratified random sample (`point_validation_dir` in config.yaml: `labeling_points.gpkg`, `sample_key.csv`, `strata_summary.csv`). Each map's probability is read at the sample points; the sample design (strata from the NRT maps, their pixel counts N_h) stays the same, so the area-weighted estimates are unbiased for every map (Stehman 2014). Estimates ± 1.96 SE at the threshold, per labelled date and for all dates combined, for labels of confidence ≥ 1 and ≥ 2.

In [ ]:
qa_dir = cfg.require("point_validation_dir")

point_long, point_table = point_validation_table(
    maps,
    labels=os.path.join(qa_dir, "labeling_points.gpkg"),   # one layer per labelled date
    key=os.path.join(qa_dir, "sample_key.csv"),
    summary=os.path.join(qa_dir, "strata_summary.csv"),
    threshold=threshold / 10000,       # probability scale
    min_confidence=(1, 2),             # one block per minimum label confidence
    outdir=os.path.join(out["validation"], "points"),   # full qaqc outputs per map + the tables
)
point_table

### Opportunistic mapping recall
Discolored polygons mapped from Planet imagery of 14 August (`validation_30km_gpkg`), against each map of that date. Without mapped non-discolored areas only recall can be computed: the share of polygons whose mean probability is ≥ the threshold, of polygons with any pixel ≥ it, and of the mapped pixels ≥ it. The per-canton breakdown is saved as well.

In [ ]:
recall_overall, recall_all = recall_table(
    maps,
    polygons=cfg.require("validation_30km_gpkg"),
    raster_name=f"{year}-08-14",   # the map of the mapping date (found by date)
    thresholds=(threshold / 10000,),
    regions=districts, region_col="ktnr",          # also per canton
    outdir=os.path.join(out["validation"], "recall"),
)
recall_overall

## Time Series

In [ ]:
# NRT, Retro and the comparison season on one month/day axis.
# Greyscale: NRT solid grey, Retro solid black, the comparison season dashed black
nrt_label, retro_label = f"{year} (Near-Real-Time)", f"{year} (Retrospective)"
nrtmm_label, compare_label = f"{year} (Retrospective, NRT min/max)", f"{compare_year} (Retrospective)"
dataset_colors = {nrt_label: "0.6", retro_label: "black", nrtmm_label: "tab:blue", compare_label: "black"}
dataset_styles = {nrt_label: "-", retro_label: "-", nrtmm_label: "-.", compare_label: "--"}

datasets = {
    nrt_label: dict(mosaic_dir=nrt_dirs["masked"], year=year),
    retro_label: dict(mosaic_dir=retro_dirs["masked"], year=year),
}
if include_nrtminmax:
    datasets[nrtmm_label] = dict(mosaic_dir=nrtmm_dirs["masked"], year=year)
datasets[compare_label] = dict(mosaic_dir=mosaics_compare, pattern=compare_pattern, year=compare_year)

comparison = plot_year_comparison(
    datasets,
    colors=dataset_colors, linestyles=dataset_styles,
    threshold=threshold,
    output_dir=out["statistics"],
    output_name=f"discoloration_{year}_nrt_vs_retro",
    max_workers=8,
)

In [ ]:
# Difference per date: retrospective minus near real time (proportion of beech forest discolored)
total = comparison[comparison.zone == "All"].pivot(index="date", columns="dataset", values="proportion")
diff = (total[retro_label] - total[nrt_label]).dropna().rename("retro - nrt")
diff.to_csv(os.path.join(out["statistics"], f"discoloration_{year}_retro_minus_nrt.csv"))
print(diff.round(4).to_string())

### Per biogeographic region
(A) Switzerland and (B) one panel per biogeographic region in its colour; the line style is the run: NRT dotted, Retro solid, the comparison season dashed. Every dataset is counted per region here, so this takes a while.

In [ ]:
region_styles = {nrt_label: ":", retro_label: "-", nrtmm_label: "-.", compare_label: "--"}

by_region = plot_year_comparison(
    datasets,
    zones=biogeo_regions, zone_field="DERegionNa",   # regions = All + every DERegionNa
    region_names=BIOGEO_NAMES_EN,                   # German -> English names
    region_colors=BIOGEO_COLORS,                    # colour per region (and panel order)
    linestyles=region_styles,
    layout="summary",
    threshold=threshold,
    output_dir=out["statistics"],
    output_name=f"discoloration_{year}_nrt_vs_retro_biogeo",
    max_workers=8,
)

## Interpolation Example
One pixel's observations and RBF interpolation: the comparison season (retrospective) and this season, near real time and retrospective.

In [ ]:
from datetime import date

force_directory = cfg["force_dir"]
force_directory_compare = cfg.require("force_2018_dir")
as_of = min(date.today(), date(year, 12, 31))   # the near-real-time curve as at the end of the season

tile, vi = "X0056_Y0061", "EVI"
row, col = 2425, 2156  # pixel within the tile (None, None = pick one with enough observations)

with plt.rc_context(rc_params(1.2)):
    fig, (ax_cmp, ax_year) = plt.subplots(2, 1, figsize=(15, 11))
plot_pixel(
    os.path.join(force_directory_compare, tile, f"{compare_year}-{compare_year}_001-365_HL_TSA_SEN2L_{vi}_TSS.tif"),
    row=row, col=col, rbf_sigma=(10, 20, 30, 50), rbf_cutoff=0.95, int_day=5,
    lag_days=None,  # retrospective ensemble only
    despike=True, above_noise=3.0, below_noise=1.0,
    title=f"{compare_year}, {tile} {vi}", ax=ax_cmp, component_data="retrospective",
)
plot_pixel(
    os.path.join(force_directory, tile, f"{year}0101-{year}1231_001-365_HL_TSA_SEN2L_{vi}_TSS.tif"),
    row=row, col=col, rbf_sigma=(10, 20, 30), rbf_cutoff=0.95, int_day=5,
    lag_days=5, as_of=as_of,           # near real time as in DiscoCH_NRT, with the retrospective curve
    component_data="nrt",              # kernel curves near-real-time ("retrospective" = all data)
    despike=False,
    title=f"{year}, {tile} {vi}", ax=ax_year,
)
# print_effective_bandwidth=True prints how much smoothing each date really got
os.makedirs(out["figures"], exist_ok=True)
fig.savefig(os.path.join(out["figures"], f"interpolation_{tile}_{vi}_r{row}_c{col}.png"), dpi=200, bbox_inches="tight")
plt.show()